# OpenPose Analysis Pipeline From JSON

This notebook reads pre-extracted OpenPose JSON data, converts it into a unified NumPy representation, and runs the same downstream gait analysis stages used in the existing workflow.

It does not perform any video processing.

In [ ]:
import glob
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.signal import butter, filtfilt, find_peaks

PROJECT_ROOT = "/home/projects/sipl-prj10496/project_files/projectA_repo"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.skeleton_tracking import OpenPoseIndices, filter_and_track_person

try:
    from src.libs.openPose_classes import KeypointPostProcessor
except ImportError:
    from src.models.openPose_video_process import KeypointPostProcessor

In [ ]:
# OpenPose JSON loader and BODY_25 parsing helpers.
# The loader accepts either a single JSON file or a directory of per-frame JSON files.
# It parses person['body']['keypoints'] and person['body']['scores'] into (J, 3) arrays.

NUM_KEYPOINTS = 25
DEFAULT_CONFIDENCE = 0.0

OPENPOSE_JOINT_MAP = {
    'mid_hip': OpenPoseIndices.MID_HIP,
    'right_hip': OpenPoseIndices.R_HIP,
    'right_knee': OpenPoseIndices.R_KNEE,
    'right_ankle': OpenPoseIndices.R_ANKLE,
    'left_hip': OpenPoseIndices.L_HIP,
    'left_knee': OpenPoseIndices.L_KNEE,
    'left_ankle': OpenPoseIndices.L_ANKLE,
}

def _pad_or_trim_keypoints(keypoints, num_keypoints=NUM_KEYPOINTS):
    keypoints = np.asarray(keypoints, dtype=np.float32)
    if keypoints.ndim != 2 or keypoints.shape[1] != 3:
        raise ValueError("Expected keypoints with shape (num_keypoints, 3).")

    if keypoints.shape[0] == num_keypoints:
        return keypoints

    padded = np.zeros((num_keypoints, 3), dtype=np.float32)
    limit = min(num_keypoints, keypoints.shape[0])
    padded[:limit] = keypoints[:limit]
    return padded

def _parse_openpose_body(person):
    body = person.get('body', {})
    keypoints_raw = np.asarray(body.get('keypoints', []), dtype=np.float32)
    scores_raw = np.asarray(body.get('scores', []), dtype=np.float32)

    if keypoints_raw.size == 0:
        return np.zeros((NUM_KEYPOINTS, 3), dtype=np.float32)

    if keypoints_raw.ndim == 1:
        if keypoints_raw.size % 3 == 0:
            keypoints_raw = keypoints_raw.reshape(-1, 3)
        elif keypoints_raw.size % 2 == 0:
            keypoints_raw = keypoints_raw.reshape(-1, 2)
        else:
            raise ValueError("OpenPose keypoints must be a flat 2D or 3D sequence.")

    if keypoints_raw.shape[1] == 2:
        if scores_raw.size == keypoints_raw.shape[0]:
            keypoints = np.column_stack([keypoints_raw, scores_raw])
        else:
            keypoints = np.column_stack([keypoints_raw, np.full(keypoints_raw.shape[0], DEFAULT_CONFIDENCE, dtype=np.float32)])
    elif keypoints_raw.shape[1] == 3:
        keypoints = keypoints_raw.copy()
        if scores_raw.size == keypoints.shape[0]:
            keypoints[:, 2] = scores_raw
    else:
        raise ValueError("OpenPose keypoints must have 2 or 3 columns.")

    return _pad_or_trim_keypoints(keypoints, NUM_KEYPOINTS)

def load_openpose_json_source(source_path):
    """Load one OpenPose JSON file or a directory of per-frame JSON files."""
    if os.path.isdir(source_path):
        json_paths = sorted(glob.glob(os.path.join(source_path, '*.json')))
        if not json_paths:
            raise FileNotFoundError(f'No JSON files found in directory: {source_path}')
        frame_payloads = []
        for json_path in json_paths:
            with open(json_path, 'r', encoding='utf-8') as handle:
                frame_payloads.append(json.load(handle))
    else:
        with open(source_path, 'r', encoding='utf-8') as handle:
            payload = json.load(handle)
        if isinstance(payload, list):
            frame_payloads = payload
        elif isinstance(payload, dict) and 'frames' in payload:
            frame_payloads = payload['frames']
        else:
            frame_payloads = [payload]

    all_frames = []
    for frame in frame_payloads:
        if isinstance(frame, dict) and 'people' in frame:
            people = frame['people']
        elif isinstance(frame, dict) and 'persons' in frame:
            people = frame['persons']
        else:
            people = []

        frame_people = []
        for person in people:
            frame_people.append(_parse_openpose_body(person))
        all_frames.append(frame_people)

    return all_frames

def track_target_person_sequence(frame_people_list, cam_num=3, min_conf=0.3):
    """Track the same target person across OpenPose frames and return a unified array."""
    tracked_frames = []
    last_known_hip_x = None

    for frame_people in frame_people_list:
        if not frame_people:
            tracked_frames.append(np.zeros((NUM_KEYPOINTS, 3), dtype=np.float32))
            continue

        best_person_idx = filter_and_track_person(
            all_keypoints=[person.tolist() for person in frame_people],
            last_known_hip_x=last_known_hip_x,
            cam_num=cam_num,
            model_type='openpose',
            use_roi_filter=True,
            min_conf=min_conf,
        )

        if best_person_idx == -1:
            tracked_frames.append(np.zeros((NUM_KEYPOINTS, 3), dtype=np.float32))
            continue

        selected = np.asarray(frame_people[best_person_idx], dtype=np.float32)
        tracked_frames.append(selected)
        last_known_hip_x = float(selected[OpenPoseIndices.MID_HIP][0])

    return np.asarray(tracked_frames, dtype=np.float32)

In [ ]:
# Load NL129_3_3 and HC65_3 OpenPose JSON data.
# Update the HC65 path to the exact JSON file name in your output directory.

JSON_SPECS = {
    'NL129_3_3': {
        'json_path': '/home/projects/sipl-prj10496/project_files/data/openpose_output/20260530_133716/NL129_3_3_keypoints_20260530_133716_0_to_end.json',
        'cam_num': 3,
    },
    'HC65_3': {
        'json_path': '/home/projects/sipl-prj10496/project_files/data/openpose_output/20260530_143600/YOUR_JSON_NAME.json',
        'cam_num': 3,
    },
}

openpose_sequences = {}
for patient_id, spec in JSON_SPECS.items():
    json_path = spec['json_path']
    if not os.path.exists(json_path):
        print(f'Skipping {patient_id} because the JSON file was not found: {json_path}')
        continue

    frame_people_list = load_openpose_json_source(json_path)
    keypoints_array = track_target_person_sequence(
        frame_people_list=frame_people_list,
        cam_num=spec['cam_num'],
        min_conf=0.3,
    )

    openpose_sequences[patient_id] = {
        'json_path': json_path,
        'frame_people_list': frame_people_list,
        'keypoints_array': keypoints_array,
    }

    print(f'{patient_id}: loaded {keypoints_array.shape[0]} frames with shape {keypoints_array.shape[1:]}')
    display(pd.DataFrame({'frames': [keypoints_array.shape[0]], 'keypoints': [keypoints_array.shape[1]], 'dimensions': [keypoints_array.shape[2]]}))

In [ ]:
# Spatial ROI filtering, anatomical validation, missing keypoint interpolation,
# intermediate saving placeholder, and Butterworth low-pass filtering.

FPS = 30.0
CUT_OFF_FREQUENCY = 6.0
FILTER_ORDER = 4
post_processor = KeypointPostProcessor(fs=FPS, conf_threshold=0.2)

def process_openpose_sequence(keypoints_array, patient_id):
    """Run the OpenPose sequence through the same downstream analysis stages."""
    if keypoints_array.size == 0:
        return np.zeros((0, NUM_KEYPOINTS, 3), dtype=np.float32), np.zeros((0, NUM_KEYPOINTS, 3), dtype=np.float32), None

    print(f'[{patient_id}] Filling missing keypoints with interpolation.')
    keypoints_filled = post_processor.fill_missing_keypoints(keypoints_array)

    # Placeholder for saving the intermediate raw continuous file.
    output_dir = os.path.join(PROJECT_ROOT, 'project_files', 'outputs', 'openpose_continuous')
    os.makedirs(output_dir, exist_ok=True)
    raw_continuous_path = os.path.join(output_dir, f'{patient_id}_raw_continuous.npy')
    np.save(raw_continuous_path, keypoints_filled)
    print(f'[{patient_id}] Saved intermediate continuous file to: {raw_continuous_path}')

    print(f'[{patient_id}] Applying Butterworth low-pass filtering.')
    keypoints_filtered = post_processor.temporal_filter(keypoints_filled, fc=CUT_OFF_FREQUENCY, order=FILTER_ORDER)

    return keypoints_filled, keypoints_filtered, raw_continuous_path

processed_sequences = {}
for patient_id, payload in openpose_sequences.items():
    keypoints_filled, keypoints_filtered, raw_continuous_path = process_openpose_sequence(
        payload['keypoints_array'],
        patient_id,
    )
    processed_sequences[patient_id] = {
        'keypoints_filled': keypoints_filled,
        'keypoints_filtered': keypoints_filtered,
        'raw_continuous_path': raw_continuous_path,
    }

In [ ]:
# Gait event detection: heel strikes and step times.
# This uses the filtered ankle trajectories and reports one event table per subject.

def detect_heel_strikes_and_step_times(keypoints_filtered, patient_id, fps=FPS):
    if keypoints_filtered.size == 0:
        return pd.DataFrame(columns=['video_id', 'frame_index', 'global_step_index', 'side', 'event_type', 'step_time_s', 'valid'])

    left_ankle_y = keypoints_filtered[:, OpenPoseIndices.L_ANKLE, 1]
    right_ankle_y = keypoints_filtered[:, OpenPoseIndices.R_ANKLE, 1]

    min_distance = max(1, int(fps * 0.35))
    left_peaks, _ = find_peaks(-left_ankle_y, distance=min_distance, prominence=5)
    right_peaks, _ = find_peaks(-right_ankle_y, distance=min_distance, prominence=5)

    events = []
    for frame_index in left_peaks:
        events.append({'video_id': patient_id, 'frame_index': int(frame_index), 'side': 'left', 'event_type': 'heel_strike'})
    for frame_index in right_peaks:
        events.append({'video_id': patient_id, 'frame_index': int(frame_index), 'side': 'right', 'event_type': 'heel_strike'})

    events = sorted(events, key=lambda row: row['frame_index'])

    step_rows = []
    previous_frame_index = None
    for global_step_index, event in enumerate(events, start=1):
        if previous_frame_index is None:
            step_time_s = np.nan
            valid = False
        else:
            step_time_s = (event['frame_index'] - previous_frame_index) / fps
            valid = True

        step_rows.append({
            'video_id': patient_id,
            'frame_index': event['frame_index'],
            'global_step_index': global_step_index,
            'side': event['side'],
            'event_type': event['event_type'],
            'step_time_s': step_time_s,
            'valid': valid,
        })
        previous_frame_index = event['frame_index']

    events_df = pd.DataFrame(step_rows)
    print(f'[{patient_id}] Detected {len(events_df)} heel strike events.')
    return events_df

gait_event_tables = {}
for patient_id, payload in processed_sequences.items():
    events_df = detect_heel_strikes_and_step_times(payload['keypoints_filtered'], patient_id, fps=FPS)
    gait_event_tables[patient_id] = events_df
    display(events_df.head(10))

In [ ]:
# Final summary for both subjects.
for patient_id, events_df in gait_event_tables.items():
    print(f'=== {patient_id} summary ===')
    if events_df.empty:
        print('No gait events were detected.')
        continue

    display(events_df)
    valid_events = events_df[events_df['valid']].copy()
    if not valid_events.empty:
        print(f